In [1]:
from freelance_finance_dl.dataloader import FinanceTransactionDataset, get_data_loader
import pandas as pd

In [2]:
raw_data_path = "../data/budgetwise_finance_dataset.csv"

df = pd.read_csv(raw_data_path)

print("Raw dataset shape:", df.shape)

df.head()

Raw dataset shape: (15900, 9)


,transaction_id,user_id,date,transaction_type,category,amount,payment_mode,location,notes
0,T4999,U018,2023-04-25,Expense,Educaton,3888,card,Ahmedabad,Movie tickets
1,T12828,U133,08/05/2022,Expense,rent,649,NaN,Hyderabad,asdfgh
2,T7403,U091,31-12-23,Income,Freelance,13239,Csh,BAN,Books
3,T12350,U097,NaN,Expense,Fod,6299,Bank Transfer,AHMEDABAD,Electricity bill
4,T7495,U088,10/28/2022,Expense,entertainment,2287,CARD,Hyderabad,NaN


In [3]:
print("Columns in raw dataset:")

for col in df.columns:
    print(col)

Columns in raw dataset:
transaction_id
user_id
date
transaction_type
category
amount
payment_mode
location
notes


In [4]:
sequence_length = 5

dataset = FinanceTransactionDataset(
    csv_file=raw_data_path,
    sequence_length=sequence_length
)

print("Number of sequence samples:", len(dataset))

Number of sequence samples: 4227


In [5]:
features, target = dataset[0]

print("Feature tensor:")
print(features)

print("\nTarget:")
print(target)

print("\nFeature shape:", features.shape)
print("Target shape:", target.shape)

Feature tensor:
tensor([[0.0000e+00, 9.0000e+00, 9.0000e+00, 4.1780e+03],
        [0.0000e+00, 4.2000e+01, 1.0000e+00, 6.7900e+02],
        [0.0000e+00, 1.0000e+00, 1.8000e+01, 9.3760e+03],
        [0.0000e+00, 5.0000e+00, 1.6000e+01, 9.4840e+03],
        [0.0000e+00, 5.0000e+00, 1.6000e+01, 9.4840e+03]])

Target:
tensor(5385.)

Feature shape: torch.Size([5, 4])
Target shape: torch.Size([])


Each dataset sample is a sequence of previous transactions.

For this demo:

- Sequence length = 5
- Number of features per transaction = 4

The input shape is therefore:

`[5, 4]`

This means the model sees 5 previous transactions, where each transaction contains:

1. encoded transaction type
2. encoded category
3. encoded payment mode
4. transaction amount

The target is the next transaction amount after the sequence.

In [6]:
loader = get_data_loader(
    csv_file=raw_data_path,
    batch_size=32,
    sequence_length=5,
    shuffle=True
)

In [7]:
batch_features, batch_targets = next(iter(loader))

print("Batch feature shape:", batch_features.shape)
print("Batch target shape:", batch_targets.shape)

Batch feature shape: torch.Size([32, 5, 4])
Batch target shape: torch.Size([32])


In [8]:
print("First sequence in batch:")
print(batch_features[0])

print("\nTarget for first sequence:")
print(batch_targets[0])

First sequence in batch:
tensor([[ 1.0000e+00,  2.6000e+01,  6.0000e+00,  1.3928e+04],
        [ 0.0000e+00,  1.0000e+00, -1.0000e+00,  8.9700e+03],
        [ 0.0000e+00,  3.8000e+01,  1.0000e+01,  3.2930e+03],
        [ 0.0000e+00,  2.1000e+01,  1.3000e+01,  3.2180e+03],
        [ 0.0000e+00,  6.0000e+00,  2.0000e+00,  7.9400e+02]])

Target for first sequence:
tensor(794.)


This notebook demonstrates that the project package installs correctly and that the dataloader matches the project proposal.

The project proposal describes modeling financial transaction histories as temporal sequences. The updated dataloader now returns sequences of previous transactions instead of single transaction rows.

For each sample:

`previous 5 transactions → next transaction amount`

This format is appropriate for a future LSTM or other sequence-based model.